In [11]:
import pandas as pd

In [12]:
df = pd.read_excel("CategorizedArticles (17).xlsx") 

In [13]:
df

,Author 1: Relevant Article? (Yes/No),Author 2: Relevant Article? (Yes/No),date_published,title,keywords,abstract,PMID,authors,journal,citation,Relevant,category,URL,Downloaded,Text
0,No,Y,2016 Jun,[Safe and Effective Analgesia with Bilateral C...,"['Anesthesia, Epidural', 'Aortic Aneurysm, Abd...",A patient with Marfan syndrome underwent emerg...,27483660,"['Katakura Y', 'Sakurai A', 'Endo M', 'Hamada ...",Masui. The Japanese journal of anesthesiology,"Katakura, Y., Sakurai, A., Endo, M., Hamada, T...",1.0,marfan,Not available,False,Text not available
1,No,Y,2019 May,Intracranial hypertension in a patient with Cl...,"['Acetazolamide/*administration & dosage', 'An...",No abstract available,30995704,"['Chia XX', 'Cordato D', 'Venkat A', 'Spicer ST']","Nephrology (Carlton, Vic.)","Chia, XX., Cordato, D., Venkat, A., Spicer, ST...",1.0,"pseudotumor cerebri, other",https://libkey.io/libraries/731/articles/29959...,False,Text not available
2,No,Y,2019 May,Intracranial hypertension in a patient with Cl...,"['Acetazolamide/*administration & dosage', 'An...",No abstract available,30995704,"['Chia XX', 'Cordato D', 'Venkat A', 'Spicer ST']","Nephrology (Carlton, Vic.)","Chia, XX., Cordato, D., Venkat, A., Spicer, ST...",1.0,"pseudotumor cerebri, other",https://libkey.io/libraries/731/articles/29959...,False,Text not available
3,Y,No,2020 Jan,Parameters Related to Lumbar Puncture Do not A...,"['Headache', 'Lumbar puncture', 'Post-dural pu...",BACKGROUND: Post-dural puncture headache (PDPH...,31562971,"['Ljubisavljevic S', 'Trajkovic JZ', 'Ignjatov...",World neurosurgery,"Ljubisavljevic, S., Trajkovic, JZ., Ignjatovic...",1.0,other,https://libkey.io/libraries/731/articles/34577...,False,Text not available
4,Y,No,2021 Jun 3,Extra-axial haemorrhage in a patient with Alpo...,"['anaesthesia', 'pregnancy']",Extra-axial haemorrhage following epidural ana...,34083183,"['Wijayanayaka S', 'Guha A', 'Sivanesan K', 'V...",BMJ case reports,"Wijayanayaka, S., Guha, A., Sivanesan, K., Vee...",1.0,hematoma,Not available,False,Text not available
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,No,No,2008 Feb,Posterior reversible encephalopathy syndrome d...,"['Adult', 'Blindness, Cortical/diagnosis/*etio...",Posterior reversible encephalopathy syndrome (...,18250139,"['Leroux G', 'Sellam J', 'Costedoat-Chalumeau ...",Lupus,"Leroux, G., Sellam, J., Costedoat-Chalumeau, N...",NaN,other,https://libkey.io/libraries/731/articles/53985...,False,Text not available
211,No,No,2018 Nov,Subdural Tension on the Brain in Patients with...,"['Chronic subdural hematoma', 'Hemiparesis', '...",BACKGROUND: Hemiparesis is a major symptom of ...,30075268,"['Tomita Y', 'Yamada SM', 'Yamada S', 'Matsuno...",World neurosurgery,"Tomita, Y., Yamada, SM., Yamada, S., Matsuno, ...",NaN,other,https://libkey.io/libraries/731/articles/21729...,False,Text not available
212,No,No,2022 Oct 29,Occipital Headache Evaluation and Rates of Mig...,"['Headache', 'Interventional Pain', 'Migraine'...",OBJECTIVE: Diagnosis of patients with occipita...,35595240,"['Love SM', 'Hopkins BD', 'Migdal CW', 'Schust...","Pain medicine (Malden, Mass.)","Love, SM., Hopkins, BD., Migdal, CW., Schuster...",NaN,other,https://libkey.io/libraries/731/articles/52616...,False,Text not available
213,No,No,2016 May,Spontaneous intracranial hypotension following...,"['Anesthesia, Epidural/*adverse effects', 'App...",We report a case of refractory spontaneous int...,26939569,"['An X', 'Wu S', 'He F', 'Li C', 'Fang X']",Acta anaesthesiologica Scandinavica,"An, X., Wu, S., He, F., Li, C., Fang, X. (2016...",NaN,spontanous,https://libkey.io/libraries/731/articles/58197...,False,Text not available


In [14]:
# TODO change the coln name from Catoegory to Categories
df['category'] = df['category'].str.split(', ')
df_exploded = df.explode('category')

# get categories
categories = df_exploded['category'].unique()

In [15]:
print(categories)
print(len(categories))

['marfan' 'pseudotumor cerebri' 'other' 'hematoma' 'autoimmune'
 'connective tissue' 'lupus' 'meningitis' 'spontanous']
9


In [18]:
categories = df['category'].unique()
user_question = "Does a diagnosis of a connective tissue disease contribute to a post-dural spinal puncture headache?"

In [29]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_openai import AzureChatOpenAI
import os
from langchain.chains.llm import LLMChain
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents.stuff import StuffDocumentsChain
from langchain.chains import MapReduceDocumentsChain, ReduceDocumentsChain
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

from langchain_community.document_loaders import DataFrameLoader

# remove this later
CHAT = AzureChatOpenAI(
    azure_endpoint="https://nlp-ai-svc.openai.azure.com/",
    openai_api_version="2024-02-15-preview",
    azure_deployment="ChatGPT16k",
    openai_api_type="azure",
    temperature=0,
    model_name="gpt-35-turbo-16",
    openai_api_key=os.environ.get("OPENAI_API_KEY")
)


# CATEGORIZATION_SUMMARY_SYSTEM_TEMPLATE = """I am considering a scoping reivew project for given category I want to understand how the existing literature guides. Write a single detailed summary based on the full texts of the article for each category."""

SUMMARIZE_CATEGORY_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. All of the article summaries below were assigned the category {category}. Write a single paragrph final summary of the following journal article summaries, focusing on my question. Liberally use APA-style in-text citations throughout the paragraph, citing the summarized articles. The article summaries are separated by '---'"

SUMMARIZE_ARTICLE_TEMPLATE = "I am working  on a scoping review to address this question: {question}\n\n Currently, I am summarizing articles by expert-defined categories. The article below was assigned the category {category}. Write a single paragrph summarizing the artcle, focusing on my question. The entire paragraph will be cited using only the article being summarized. Write the paragaraph so providing only the citation for the single article will be appropriate."



HUMAN_TEMPLATE = """
Content to summarize:
{content}
"""

category_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_CATEGORY_TEMPLATE)
article_system_message_prompt = SystemMessagePromptTemplate.from_template(SUMMARIZE_ARTICLE_TEMPLATE)

human_message_prompt = HumanMessagePromptTemplate.from_template(HUMAN_TEMPLATE)

category_summary_chat_prompt = ChatPromptTemplate.from_messages([category_system_message_prompt, human_message_prompt])
article_summary_chat_prompt = ChatPromptTemplate.from_messages([article_system_message_prompt, human_message_prompt])


In [22]:

output = []
for current_category in categories:
    filtered_rows = df[(df['category'] == current_category)]

    article_summaries = []
    for _, row in filtered_rows.iterrows():
        result = CHAT.invoke(article_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content="APA Citation: " + row.APA_Citation + "\n\nArticle text:\n\n" +row.Text,
        ).to_messages())

        summary_to_keep = f"APA Citation: {row.APA_Citation}\n\n Summary: {result.content}\n\n --- "
        # print(summary_to_keep)
        article_summaries.append(summary_to_keep)

    text_to_summarize = "\n\n".join(article_summaries)
   # text_to_summarize = f"Category of current articles: {current_category} \n\n" + text_to_summarize
    
    result = CHAT.invoke(category_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content=text_to_summarize
        ).to_messages())
    print(result.content)
    output.append(result.content)
    print("************")
    

The articles reviewed provide insights into the potential relationship between connective tissue diseases and post-dural spinal puncture headache. Wijayanayaka et al. (2021) present a case of an Alport syndrome patient who developed an extra-axial hemorrhage following epidural anesthesia, suggesting that subdural hemorrhage should be considered as a potential complication in patients with connective tissue diseases. Martin and Neilson (2014) highlight the barriers physicians face in diagnosing and managing patients with Ehlers-Danlos Syndrome (EDS), indicating a need for improved diagnosis and care for patients with connective tissue diseases. Londhey (2015) discusses the diverse clinical manifestations of neuropathies associated with systemic lupus erythematosus (SLE), providing insights into the neurological disorders associated with connective tissue diseases. While these articles do not directly address the specific question of whether a diagnosis of connective tissue disease contr

In [ ]:
# use abtract when text is not available.
df['Text'] = df.apply(lambda row: row['abstract'] if row['Text'] == 'Text not available' else row['Text'], axis=1)


In [24]:
def summarize_article_in_chunks(article_text):
    # Splitting the article text into manageable chunks
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=13000, chunk_overlap=1000)
    texts = text_splitter.create_documents([article_text])

    # Template for the initial summarization of the first chunk
    initial_summary_prompt = """I am working on a scoping review to address a specific question.
    I need to summarize this journal article, focusing on the given question and the article's category.
    Here is the first chunk of the journal article:

    {text}
    """

    # Template for refining the summary with each subsequent chunk
    refine_summary_prompt = """Based on the existing summary and the specific question, refine the summary with the information from the next chunk of the article.
    Current summary:

    {existing_summary}

    Next chunk of the article:
    ------------
    {text}
    ------------
    """

    # Create the initial summary for the first chunk
    summary = CHAT.invoke(initial_summary_prompt.format(text=texts[0]))

    # Iteratively refine the summary with each subsequent chunk
    for text_chunk in texts[1:]:
        summary = CHAT.invoke(refine_summary_prompt.format(existing_summary=summary, text=text_chunk))

    return summary




In [30]:
# Usage of the function within the larger script
article_summaries = []
for current_category in categories:
    filtered_rows = df[df['category'] == current_category]

    for _, row in filtered_rows.iterrows():
        article_summary = summarize_article_in_chunks(row.Text)
        formatted_summary = f"APA Citation: {row.APA_Citation}\n\n Summary: {article_summary}\n\n --- "
        article_summaries.append(formatted_summary)
        
        text_to_summarize = "\n\n".join(article_summaries)
   # text_to_summarize = f"Category of current articles: {current_category} \n\n" + text_to_summarize
    
    result = CHAT.invoke(category_summary_chat_prompt.format_prompt(
        question=user_question,
        category=current_category,
        content=text_to_summarize
        ).to_messages())
    print(current_category)
    print(result.content)
    output.append(result.content)
    print("************")



'Neurological Disorders'
A scoping review of articles related to connective tissue diseases and post-dural spinal puncture headache (PDPH) identified several relevant articles. One case report discussed a rare case of extra-axial hemorrhage in a patient with Alport syndrome following epidural anesthesia, highlighting the challenges in differentiating between PDPH and other complications (Wijayanayaka et al., 2021). Another article focused on the barriers physicians face in diagnosing and managing patients with Ehlers-Danlos Syndrome (EDS), a connective tissue disease, emphasizing the need for clinical practice guidelines and additional support and training for providers (Martin & Neilson, 2014). Additionally, a study explored the association between chronic inflammatory demyelinating polyradiculoneuropathy (CIDP), a connective tissue disease, and other conditions such as diabetes and systemic lupus erythematosus (Londhey, 2015). While these articles provide insights into the complicati

In [15]:
# Map reduce
for current_category in categories:
    # Map
    map_template = "I am working  on a scoping literature review to address this question: " + user_question + \
    "\n\nCurrently, I am summarizing articles by expert-defined categories. All of the articles below were assigned the category " + current_category + \
    """{docs}
    \n\nSummarize the provided articles, focusing on my question. Use in-text citations in APA format. The documents' metadata contains the full APA citation. 
    Use only the context of the articles, not any other information. \n\n"""
    map_prompt = PromptTemplate.from_template(map_template)
    map_chain = LLMChain(llm=CHAT, prompt=map_prompt)

    reduce_template = "I am working  on a scoping literature review to address this question: " + user_question + \
    "\n\nCurrently, I need a final summary of all the documents below. All of the article summaries below were assigned the category " + current_category + \
    """The following is set of summaries of articles:
    {docs}
    Take these and distill it into a final, consolidated summary paragraph appropriate for insertion into an academic journal article. Maintain in-text citations in the APA format. I'll add the full bibliography later. Use only the context of the articles, not any other information."""
    reduce_prompt = PromptTemplate.from_template(reduce_template)
    reduce_chain = LLMChain(llm=CHAT, prompt=reduce_prompt)

    # Takes a list of documents, combines them into a single string, and passes this to an LLMChain
    combine_documents_chain = StuffDocumentsChain(
        llm_chain=reduce_chain, document_variable_name="docs"
    )

    # Combines and iteratively reduces the mapped documents
    reduce_documents_chain = ReduceDocumentsChain(
        # This is final chain that is called.
        combine_documents_chain=combine_documents_chain,
        # If documents exceed context for `StuffDocumentsChain`
        collapse_documents_chain=combine_documents_chain,
        # The maximum number of tokens to group documents into.
        token_max=10000,
    )

    # Combining documents by mapping a chain over them, then combining results
    map_reduce_chain = MapReduceDocumentsChain(
        # Map chain
        llm_chain=map_chain,
        # Reduce chain
        reduce_documents_chain=reduce_documents_chain,
        # The variable name in the llm_chain to put the documents in
        document_variable_name="docs",
        # Return the results of the map steps in the output
        return_intermediate_steps=False,
    )





    filtered_rows = df[(df['category'] == current_category) & (df['Text'] != "Text not available")]
    text_columns = df[['Text', 'APA_Citation']]
    loader = DataFrameLoader(text_columns, page_content_column="Text")
    docs = loader.load()
    # if not filtered_rows.empty:
    #     new_df = pd.concat([new_df, filtered_rows])
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100000, chunk_overlap=0
    )
    split_docs = text_splitter.split_documents(docs)
    print(map_reduce_chain.run(split_docs))


The scoping literature review aimed to address the question of whether a diagnosis of a connective tissue disease contributes to a post-dural spinal puncture headache. However, the provided articles did not directly address this question. Instead, they focused on various topics such as neurological disorders, chronic daily headache, Ehlers-Danlos Syndrome (EDS), intrathecal analgesia, chronic inflammatory demyelinating polyradiculoneuropathy (CIDP), systemic lupus erythematosus (SLE), and periodontitis. These articles provided insights into the phenotype of new daily persistent headache (NDPH) and its comparison to transformed chronic daily headache (T-CDH), the barriers faced by physicians in diagnosing and managing EDS, the use of intrathecal analgesia in a patient with progressive systemic sclerosis (PSS), the association between CIDP and concomitant diseases, the long-term safety and efficacy of anifrolumab in SLE patients, the association between SSc and periodontitis, the comorbi